<a href="https://colab.research.google.com/github/hamdikhasawneh/AI-sepsis/blob/main/notebooks/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/gp/Cleaned')
print(DATA_DIR.exists())

True


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/gp/Cleaned/chartevents_labevents.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    print(z.namelist())

['Cleaned/', 'Cleaned/chartevents.csv', 'Cleaned/labevents.csv']


In [ ]:
for file in DATA_DIR.iterdir():
    print(file.name)

inputevents.csv
outputevents.csv
procedureevents.csv
ingredientevents.csv
datetimeevents.csv
icustays.csv
d_items.csv
patients.csv
prescriptions.csv
admissions.csv
microbiologyevents.csv
caregiver.csv
diagnoses_icd.csv
d_labitems.csv
chartevents_labevents.zip


##Load base tables


In [ ]:
import pandas as pd

In [ ]:
patients = pd.read_csv(DATA_DIR / 'patients.csv')
admissions = pd.read_csv(DATA_DIR / 'admissions.csv')
icustays = pd.read_csv(DATA_DIR / 'icustays.csv')

## Inspect Tables

In this section, we check:
- shape
- columns
- sample rows


In [ ]:
print("patients:", patients.shape)
print("admissions:", admissions.shape)
print("icustays:", icustays.shape)

patients: (364627, 6)
admissions: (546028, 16)
icustays: (94458, 8)


In [ ]:
print("patients columns:")
print(patients.columns.tolist())
print("\nadmissions columns:")
print(admissions.columns.tolist())
print("\nicustays columns:")
print(icustays.columns.tolist())

patients columns:
['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']

admissions columns:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']

icustays columns:
['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']


In [ ]:
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [ ]:
admissions.head(10)
admissions[admissions.subject_id == 10000032]

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0


In [ ]:
icustays.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


## Creating Cohort


In [ ]:
cohort = icustays.merge(
    admissions,
    on=['subject_id', 'hadm_id'],
    how='left'
).merge(
    patients,
    on='subject_id',
    how='left'
)
cohort.head()
print("cohort shape:", cohort.shape)

cohort shape: (94458, 27)


In [ ]:
cohort.isnull().sum().sort_values(ascending=False).head(20)

,0
deathtime,83117
dod,56491
edregtime,32331
edouttime,32331
marital_status,7761
insurance,1523
discharge_location,848
language,396
outtime,14
los,14


In [ ]:
print("Duplicate rows:", cohort.duplicated().sum())

Duplicate rows: 0


### Saving Cohort

In [ ]:
output_dir = Path('/content/drive/MyDrive/mimic_iv_processed')
output_dir.mkdir(parents=True, exist_ok=True)

cohort.to_csv(output_dir / 'icu_cohort.csv', index=False)
print("Saved successfully")

Saved successfully


In [ ]:
print("Cohort shape:", cohort.shape)
print(cohort["stay_id"].nunique())
print(cohort.shape[0])
print(cohort.duplicated(subset="stay_id").sum())

Cohort shape: (94458, 27)
94458
94458
0


In [ ]:
cohort.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'gender',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'dod'],
      dtype='object')

### Converting Data-Time


In [ ]:
cohort["intime"] = pd.to_datetime(cohort["intime"])
cohort["outtime"] = pd.to_datetime(cohort["outtime"])
cohort["admittime"] = pd.to_datetime(cohort["admittime"])
cohort["dischtime"] = pd.to_datetime(cohort["dischtime"])

### Calculating Length of ICU stay in column icu_los_hours


In [ ]:
cohort["icu_los_hours"] = (cohort["outtime"] - cohort["intime"]).dt.total_seconds() / 3600
cohort["icu_los_hours"].describe()

,icu_los_hours
count,94444.000000
mean,87.120596
std,129.659365
min,0.030000
25%,26.309097
50%,47.175556
75%,92.701806
max,5433.673889


### Filtering Short ICU stays

In [ ]:
# Removing less than 4 hours
cohort = cohort[cohort["icu_los_hours"] >= 4]
cohort.shape

(93730, 28)

In [ ]:
cohort.to_csv(output_dir / "icu_cohort.csv", index=False)
print("Clean cohort saved")

Clean cohort saved


In [ ]:
cohort["anchor_age"].describe()

,anchor_age
count,93730.000000
mean,63.045610
std,16.712152
min,18.000000
25%,53.000000
50%,65.000000
75%,76.000000
max,91.000000


In [ ]:
# Filtering adult patients
cohort = cohort[cohort["anchor_age"] >= 18]
print(cohort.shape)

(93730, 28)


In [ ]:
output_dir = Path('/content/drive/MyDrive/mimic_iv_processed')
output_dir.mkdir(parents=True, exist_ok=True)

cohort.to_csv(output_dir / "icu_cohort.csv", index=False)

print("Clean ICU cohort saved")

## Identify Vital Sign ITEMIDs

In [ ]:
d_items = pd.read_csv(DATA_DIR / "d_items.csv")
print(d_items.shape)
print(d_items.columns)

(4095, 9)
Index(['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname',
       'param_type', 'lownormalvalue', 'highnormalvalue'],
      dtype='object')


In [ ]:
# Important Vital Signs
# Heart Rate
d_items[d_items["label"].str.contains("Heart Rate", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


In [ ]:
# Respiratory Rate
d_items[d_items["label"].str.contains("Respiratory", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
28,220210,Respiratory Rate,RR,chartevents,Respiratory,insp/min,Numeric,NaN,NaN
470,223985,Respiratory Pattern,Respiratory Pattern,chartevents,Pulmonary,NaN,Text,NaN,NaN
475,223990,Respiratory Effort,Respiratory Effort,chartevents,Pulmonary,NaN,Text,NaN,NaN
799,224688,Respiratory Rate (Set),Respiratory Rate (Set),chartevents,Respiratory,insp/min,Numeric,NaN,NaN
800,224689,Respiratory Rate (spontaneous),Respiratory Rate (spontaneous),chartevents,Respiratory,insp/min,Numeric,NaN,NaN
801,224690,Respiratory Rate (Total),Respiratory Rate (Total),chartevents,Respiratory,insp/min,Numeric,NaN,36.0
839,224745,Respiratory Quotient,RQ,chartevents,Respiratory,NaN,Numeric,NaN,NaN
1327,225475,Respiratory Arrest,Respiratory Arrest,procedureevents,3-Significant Events,NaN,Processes,NaN,NaN
2075,227032,Non-Operative Respiratory (RESPIRAT),Respiratory Non-Operative,chartevents,Scores - APACHE IV,NaN,Text,NaN,NaN
2090,227047,Post-Operative Respiratory (RESPIRAT),Respiratory Post-Operative,chartevents,Scores - APACHE IV,NaN,Text,NaN,NaN


In [ ]:
# Temperature
d_items[d_items["label"].str.contains("Temperature", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
337,223761,Temperature Fahrenheit,Temperature F,chartevents,Routine Vital Signs,°F,Numeric,NaN,NaN
338,223762,Temperature Celsius,Temperature C,chartevents,Routine Vital Signs,°C,Numeric,NaN,NaN
505,224027,Skin Temperature,Skin Temp,chartevents,Skin - Assessment,NaN,Text,NaN,NaN
767,224642,Temperature Site,Temp Site,chartevents,Routine Vital Signs,NaN,Text,NaN,NaN
790,224674,Changes in Temperature,Changes in Temperature,chartevents,Toxicology,NaN,Text,NaN,NaN
1814,226329,Blood Temperature CCO (C),Blood Temp CCO (C),chartevents,Routine Vital Signs,°C,Numeric,NaN,NaN
2097,227054,TemperatureF_ApacheIV,TemperatureF_ApacheIV,chartevents,Scores - APACHE IV (2),°F,Numeric,NaN,NaN
2776,228242,Pt. Temperature (BG) (SOFT),Pt. Temperature (BG) (SOFT),chartevents,Labs,NaN,Numeric,NaN,NaN
3466,229236,Cerebral Temperature (C),Cerebral T (C),chartevents,Hemodynamics,°C,Numeric,NaN,NaN


In [ ]:
# SpO2
d_items[d_items["label"].str.contains("SpO2", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
1810,226253,SpO2 Desat Limit,SpO2 Desat Limit,chartevents,Alarms,%,Numeric,NaN,NaN
3920,229862,Forehead SpO2 Sensor in Place,Forehead SpO2 Sensor in Place,chartevents,Routine Vital Signs,NaN,Checkbox,NaN,NaN


In [ ]:
# Blood Pressure
d_items[d_items["label"].str.contains("Pressure", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
6,220050,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0
7,220051,Arterial Blood Pressure diastolic,ABPd,chartevents,Routine Vital Signs,mmHg,Numeric,60.0,90.0
8,220052,Arterial Blood Pressure mean,ABPm,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN
9,220056,Arterial Blood Pressure Alarm - Low,ABP Alarm - Low,chartevents,Alarms,mmHg,Numeric,NaN,NaN
10,220058,Arterial Blood Pressure Alarm - High,ABP Alarm - High,chartevents,Alarms,mmHg,Numeric,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3947,229892,Pressure Ulcer #9- Date Indentified,Pressure Ulcer #9- Date Indentified,datetimeevents,Skin - Impairment,NaN,Date and time,NaN,NaN
3948,229893,Pressure Ulcer #10- Date Indentified,Pressure Ulcer #10- Date Indentified,datetimeevents,Skin - Impairment,NaN,Date and time,NaN,NaN
3954,229899,Left Ventricular Pressure Signal - Systolic,Left Ventricular Pressure Signal - Systolic,chartevents,Impella,mmHg,Numeric,NaN,NaN
3955,229900,Left Ventricular Pressure Signal - Diastolic,Left Ventricular Pressure Signal - Diastolic,chartevents,Impella,mmHg,Numeric,NaN,NaN


In [ ]:
d_items[d_items["label"].str.contains("saturation", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
31,220227,Arterial O2 Saturation,SaO2,chartevents,Labs,%,Numeric,NaN,NaN
36,220277,O2 saturation pulseoxymetry,SpO2,chartevents,Respiratory,%,Numeric,NaN,NaN
345,223769,O2 Saturation Pulseoxymetry Alarm - High,SpO2 Alarm - High,chartevents,Alarms,%,Numeric,NaN,NaN
346,223770,O2 Saturation Pulseoxymetry Alarm - Low,SpO2 Alarm - Low,chartevents,Alarms,%,Numeric,NaN,NaN
2004,226860,RA %O2 Saturation (PA Line),RA %O2 Saturation (PA Line),chartevents,PA Line Insertion,%,Numeric,NaN,NaN
2005,226861,ART %O2 saturation (PA Line),ART %O2 saturation (PA Line),chartevents,PA Line Insertion,%,Numeric,NaN,NaN
2006,226862,PA %O2 Saturation (PA Line),PA %O2 Saturation (PA Line),chartevents,PA Line Insertion,%,Numeric,NaN,NaN
2007,226863,PVR %O2 Saturation (PA Line),PVR %O2 Saturation (PA Line),chartevents,PA Line Insertion,dynes*sec/cm5,Numeric,NaN,NaN
2008,226865,SVR %O2 Saturation (PA Line),SVR %O2 Saturation (PA Line),chartevents,PA Line Insertion,dynes*sec/cm5,Numeric,NaN,NaN
2769,228232,PAR-Oxygen saturation,PAR-Oxygen saturation,chartevents,Routine Vital Signs,NaN,Text,NaN,NaN


In [ ]:
# Selected vital sign ITEMIDs

HEART_RATE_ID = 220045
RESP_RATE_ID = 220210
TEMP_C_ID = 223762
SPO2_ID = 220277
ABP_SYS_ID = 220050
ABP_DIA_ID = 220051
ABP_MEAN_ID = 220052
VITAL_ITEMIDS = [
    HEART_RATE_ID,
    RESP_RATE_ID,
    TEMP_C_ID,
    SPO2_ID,
    ABP_SYS_ID,
    ABP_DIA_ID,
    ABP_MEAN_ID
]
print(VITAL_ITEMIDS)

[220045, 220210, 223762, 220277, 220050, 220051, 220052]


## Extract Vital Signs from chartevents


In [ ]:
chartevents_zip_path = DATA_DIR / "chartevents_labevents.zip"
print(chartevents_zip_path.exists())

True


In [ ]:
import zipfile

with zipfile.ZipFile(chartevents_zip_path, 'r') as z:
    print(z.namelist())

['Cleaned/', 'Cleaned/chartevents.csv', 'Cleaned/labevents.csv']


In [ ]:
import zipfile
import pandas as pd

chartevents_zip_path = DATA_DIR / "chartevents_labevents.zip"

filtered_chunks = []
chunk_size = 500000

with zipfile.ZipFile(chartevents_zip_path, "r") as z:
    with z.open("Cleaned/chartevents.csv") as f:
        for i, chunk in enumerate(pd.read_csv(f, chunksize=chunk_size)):
            chunk = chunk[
                (chunk["stay_id"].isin(stay_ids)) &
                (chunk["itemid"].isin(VITAL_ITEMIDS))
            ]

            if not chunk.empty:
                filtered_chunks.append(chunk)

            print(f"Finished chunk {i+1}")

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11
Finished chunk 12
Finished chunk 13
Finished chunk 14
Finished chunk 15
Finished chunk 16
Finished chunk 17
Finished chunk 18
Finished chunk 19
Finished chunk 20
Finished chunk 21
Finished chunk 22
Finished chunk 23
Finished chunk 24
Finished chunk 25
Finished chunk 26
Finished chunk 27
Finished chunk 28
Finished chunk 29
Finished chunk 30
Finished chunk 31
Finished chunk 32
Finished chunk 33
Finished chunk 34
Finished chunk 35
Finished chunk 36
Finished chunk 37
Finished chunk 38
Finished chunk 39
Finished chunk 40
Finished chunk 41
Finished chunk 42
Finished chunk 43
Finished chunk 44
Finished chunk 45
Finished chunk 46
Finished chunk 47
Finished chunk 48
Finished chunk 49
Finished chunk 50
Finished chunk 51
Finished chunk 52
Finished chunk 53
Finished chunk 54
Finished chunk 55
Finished chunk 56
F

In [ ]:
vitals = pd.concat(filtered_chunks, ignore_index=True)

print("Vitals shape:", vitals.shape)
vitals.head()

NameError: name 'pd' is not defined

In [ ]:
vitals = vitals[[
    "subject_id",
    "hadm_id",
    "stay_id",
    "charttime",
    "itemid",
    "valuenum",
    "valueuom"
]]

vitals["charttime"] = pd.to_datetime(vitals["charttime"])

vitals.head()

,subject_id,hadm_id,stay_id,charttime,itemid,valuenum,valueuom
0,10000032,29079034,39553978,2180-07-23 14:12:00,220045,91.0,bpm
1,10000032,29079034,39553978,2180-07-23 14:12:00,220210,24.0,insp/min
2,10000032,29079034,39553978,2180-07-23 14:13:00,220277,98.0,%
3,10000032,29079034,39553978,2180-07-23 14:30:00,220045,93.0,bpm
4,10000032,29079034,39553978,2180-07-23 14:30:00,220210,21.0,insp/min


In [ ]:
vitals.to_csv(output_dir / "vitals_filtered.csv", index=False)

print("Filtered vitals saved successfully")

Filtered vitals saved successfully


In [ ]:
vitals.shape

(35596303, 7)

In [ ]:
itemid_to_label = {
    220045: "heart_rate",
    220210: "resp_rate",
    223762: "temp_c",
    220277: "spo2",
    220050: "abp_sys",
    220051: "abp_dia",
    220052: "abp_mean"
}

vitals["feature_name"] = vitals["itemid"].map(itemid_to_label)
vitals.head()

,subject_id,hadm_id,stay_id,charttime,itemid,valuenum,valueuom,feature_name
0,10000032,29079034,39553978,2180-07-23 14:12:00,220045,91.0,bpm,heart_rate
1,10000032,29079034,39553978,2180-07-23 14:12:00,220210,24.0,insp/min,resp_rate
2,10000032,29079034,39553978,2180-07-23 14:13:00,220277,98.0,%,spo2
3,10000032,29079034,39553978,2180-07-23 14:30:00,220045,93.0,bpm,heart_rate
4,10000032,29079034,39553978,2180-07-23 14:30:00,220210,21.0,insp/min,resp_rate


In [ ]:
vitals["feature_name"].isnull().sum()

np.int64(0)

In [ ]:
vitals["charttime_hour"] = vitals["charttime"].dt.floor("h")
vitals.head()

,subject_id,hadm_id,stay_id,charttime,itemid,valuenum,valueuom,feature_name,charttime_hour
0,10000032,29079034,39553978,2180-07-23 14:12:00,220045,91.0,bpm,heart_rate,2180-07-23 14:00:00
1,10000032,29079034,39553978,2180-07-23 14:12:00,220210,24.0,insp/min,resp_rate,2180-07-23 14:00:00
2,10000032,29079034,39553978,2180-07-23 14:13:00,220277,98.0,%,spo2,2180-07-23 14:00:00
3,10000032,29079034,39553978,2180-07-23 14:30:00,220045,93.0,bpm,heart_rate,2180-07-23 14:00:00
4,10000032,29079034,39553978,2180-07-23 14:30:00,220210,21.0,insp/min,resp_rate,2180-07-23 14:00:00


In [ ]:
vitals_hourly = (
    vitals.groupby(["stay_id", "charttime_hour", "feature_name"])["valuenum"]
    .mean()
    .reset_index()
)

vitals_hourly.head()

NameError: name 'vitals' is not defined